In [2]:
# Pull categorization and examples from: https://www.handalyzer.com/card_categories_guide.html

# Setup

In [ ]:
import pandas as pd
import os
import utils
from google import genai
import plotnine as pn
import json
import requests
import time
import re

client = genai.Client()

In [31]:
# Load the most recent pull of card data
data = utils.load_most_recent_pull()

Loading most recent ygo_daily file...
Loading file: ../Data/Auto Pulls\ygo_daily_2025-07-30_20-51-34.csv


## Function Definitions

In [5]:
def make_prompt(card_name):

    # Get the card text from the dataset
    card_text = data.loc[data['name'] == card_name, 'desc']
    card_text = card_text.iloc[0]

    # Define the prompt for the LLM
    prompt = f"""
    You are given the full text of a Yu-Gi-Oh! card and are an expert researcher on YuGiOh card effects. Classify the card into one of the following categories based on its effect activation condition and strategic role as defined in “Understanding Yu-Gi-Oh! Card Categories - Handalyzer”:

    - Starter: A card that can begin your main combo or main line of play, usually by itself. You can open with it and make meaningful progress towards your deck's main win condition, often not needing to pair it with another specific card.
    - Handtrap: A card that disrupts the opponent, activated from the hand during either player’s turn (especially the opponent's). Its main role is to interact with or disrupt your opponent while not on the field.
    - Extender: A card that continues or extends your plays, but typically requires something to have started already. Usually helps push through interruptions or get more value after your main combo starts.
    - Board Breaker: A card used to eliminate or answer established threats on your opponent's field. Often most useful going second, able to remove, negate, or otherwise neutralize your opponent’s set-up.
    - Garnet: A card included in a deck only to facilitate a combo/engine, but is undesirable to draw; it’s usually a “necessary brick”—required for combos but almost never wanted in your opening hand.
    - None: If the card does not fit any of these categories.

    **Output format:**
    Category: <One of [Starter, Handtrap, Extender, Board Breaker, Garnet, None]>
    Justification: <Concise explanation, referencing effect activation condition and strategic role>

    Use the following examples as a guide:

    Example 1  
    Card Text: “When a card or effect is activated that includes any of these effects (Quick Effect): You can discard this card; negate that effect.
    ● Add a card from the Deck to the hand.
    ● Special Summon from the Deck.
    ● Send a card from the Deck to the GY.
    You can only use this effect of "Ash Blossom & Joyous Spring" once per turn.”  
    ```
    Category: Handtrap  
    Justification: This effect can be activated from the hand during either player's turn to disrupt the opponent’s searches, fitting the definition of a reactive, disruptive handtrap.
    ```

    Example 2  
    Card Text: “While you control a "Rescue-ACE" monster other than "Rescue-ACE Hydrant", your opponent's monsters cannot target this card for attacks, also your opponent cannot target this card with card effects. You can only use each of the following effects of "Rescue-ACE Hydrant" once per turn. You can activate a Quick-Play Spell, or Trap Card, that was Set by the effect of your "Rescue-ACE" card, the turn it was Set. During your Main Phase: You can add 1 "Rescue-ACE" monster from your Deck to your hand, except "Rescue-ACE Hydrant".”  
    ```
    Category: Starter  
    Justification: This card enables a strong opening by Special Summoning itself and searching key combo pieces, making it an ideal first move to begin plays.
    ```

    Example 3  
    Card Text: “If you control an "Exosister" monster: You can Special Summon this card from your hand, then if you control "Exosister Stella", you gain 800 LP. If your opponent moves a card(s) out of either GY (except during the Damage Step): You can Special Summon from your Extra Deck, 1 "Exosister" Xyz Monster using this face-up card you control as material. (This is treated as an Xyz Summon.) You can only use each effect of "Exosister Elis" once per turn.”  
    ```
    Category: Extender  
    Justification: The card’s effect requires an already-established board (a Dragon monster) to use, allowing additional combo progression after a play has started.
    ```

    Example 4  
    Card Text: “At the end of the Battle Phase, if your opponent controls more cards than you do: You can make your opponent banish cards from their field face-down so they control the same number of cards as you do. If you control no cards, you can activate this card from your hand.”  
    ```
    Category: Board Breaker  
    Justification: This effect is used proactively to clear established threats; it does not advance your own plays but neutralizes your opponent’s board, serving as a board breaker.
    ```

    Example 5  
    Card Text: “Your opponent cannot activate monster effects in response to the activation of your Normal Trap Cards. You can only use each of the following effects of "Lovely Labrynth of the Silver Castle" once per turn. You can target 1 Normal Trap in your GY; Set it to your field, but it cannot be activated unless you control a Fiend monster. If another monster(s) leaves the field by your Normal Trap effect (except during the Damage Step): You can destroy 1 card in your opponent's hand (at random) or their field.”  
    ```
    Category: Garnet  
    Justification: This card is included only to enable another card’s effect, and is undesirable to draw otherwise, fitting the definition of a ‘Garnet’.
    ```

    Given these definitions and examples, classify the provided card.
    {card_text}
    """

    return prompt

In [6]:
def categorize_card(card_name, local=True, 
                    local_model="gemma3:latest",
                    gem_model="gemini-2.5-pro"):
    """
    Categorizes a Yu-Gi-Oh! card based on its effect activation condition and strategic role.

    Args:
        card_name (str): The name of the card to categorize.
        local (bool): If True, uses the local client; if False, uses the remote Gemini API.
        local_model (str): The model name for the local Ollama client.
        gem_model (str): The model name for the remote Gemini API.

    Returns:
     Returns:
        str: The category and justification for the card.
    """

    prompt = make_prompt(card_name)

    if local:
        # Use the local client for testing
        model_name = local_model
        url = "http://localhost:11434/api/generate"

        # Request payload
        payload = {
            "model": model_name,
            "prompt": prompt,
            "stream": False
        }

        # Send request to Ollama API
        response = requests.post(url, json=payload, timeout=300)
        response.raise_for_status()

        # Check if the response is empty
        if not response.json().get("response"):
            raise ValueError("Received an empty response from the Ollama API.")

        # Extract and print the generated response
        result = response.json()["response"]
        return result

    else:
        # Use the remote Gemini API
        response = client.models.generate_content(
            model=gem_model,
            contents=prompt
        )
        
        # Check if the response is empty
        if not response.text.strip():
            raise ValueError("Received an empty response from the Gemini API.")
        
        # Extract and return the text content from the response
        result = response.text.strip()
        return result

In [7]:
def parse_response(card, model, response, run=None):
    """
    Parses the response from the Gemini API to extract the category and justification.

    Args:
        card (str): The name of the card being categorized.
        model (str): The model used for categorization.
        response (str): The response text from the LLM.

    Returns:
        dict: A dictionary containing the card ID, name, category, justification, and card text.
    """
    
    lines = response.split('\n')
    
    if len(lines) < 2:
        raise ValueError("Response format is incorrect. Expected at least two lines.")

    # Remove "Category:" and "Justification:" prefixes if present
    category = lines[0].replace("Category:", "").strip()
    justification = ' '.join(line.replace("Justification:", "").strip() for line in lines[1:]).strip()

    new_row = pd.DataFrame({
        'card_id': [data.loc[data['name'] == card, 'id'].values[0]],
        'card_name': [card],
        'category': [category],
        'justification': [justification], 
        'card_text': [data.loc[data['name'] == card, 'desc'].values[0]],
        'model': [model],
        'run': run if run is not None else None
    })

    return new_row

## Pull Sample to Fine-Tune RoBERTa

In [8]:
# Pull a random sample based on the numeric feature tcg_months_since_release
data['release_bin'] = pd.qcut(data['tcg_months_since_rel'], q=10, duplicates='drop')

# Now perform stratified sampling based on these bins
# Pass observed=False to silence the first warning
# Explicitly drop the grouping column after sampling to silence the second warning
sample = (
    data.groupby('release_bin', group_keys=False, observed=False)
        .apply(lambda x: x.sample(frac=0.1, random_state=42), include_groups=False)
        .reset_index(drop=True)
)

# Remove the 'release_bin' column as it's no longer needed
data = data.drop(columns=['release_bin'])

# LLM Categorization

## Helper Functions

In [9]:
def reset_model_order():
    """
    Reset the ordering of the Gemini models after the timeout.
    Removes any models that have reached their maximum count.
    """
    global model_order, failed_models, last_reset, available_models
    print("Resetting model_order to all available Gemini models after 1 minute.")
    
    # Start with the original order and filter out models that are at or over the max
    available_models = [m for m in model_order_original if model_count[m] < model_count_max[m]]
    if len(available_models) < len(model_order_original):
        removed = [m for m in model_order_original if m not in available_models]
        print(f"Models {removed} have reached their maximum count. Removing them from model_order.")
    model_order = available_models.copy()
    failed_models = []
    last_reset = time.time()
    print(f"Model order has been reset to: {model_order}")


In [10]:
def try_gemini_models(card):
    """
    Try to categorize a card using available Gemini models.
    
    Returns:
        new_row (pd.DataFrame): the parsed response if successful,
                                otherwise None.
    """
    global failed_models, available_models, run
    for model in [m for m in available_models]:
        try:
            print(f"Trying Gemini API with model: {model}")
            response = categorize_card(card, local=False, gem_model=model)
            new_row = parse_response(card, model, response, run)
            model_count[model] += 1
            print(f"Gemini API success: {new_row['category'].iloc[0]}")
            return new_row
        except Exception as e:
            print(f"Gemini API failed for {card} with model {model} with error: {e}. Removing this model for this round.")
            failed_models.append(model)
    return None

## Categorization Loop

In [32]:
# Load previously categorized results if available
results = pd.read_csv("../Data/categorized_cards.csv") if os.path.exists("../Data/categorized_cards.csv") else pd.DataFrame(columns=["card_id", "card_name", "category", "justification", "card_text"])

# Loop setup
# Gemini model ordering configuration
model_order_original = [
    "gemini-2.5-pro",
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite"
]
available_models = model_order_original.copy()
model_order = model_order_original.copy()
failed_models = []
last_reset = time.time()

model_count = {model: 0 for model in model_order_original}  
model_count_max = {
    "gemini-2.5-pro": 100,
    "gemini-2.5-flash": 250, 
    "gemini-2.5-flash-lite": 1000
}

run = 3 # UPDATE EACH TIME, This variable is used to track the run number for categorization

In [ ]:
# Main categorization loop
# for idx, card in enumerate(sample['name'], start=1): # REGULAR ORDER
for idx, card in enumerate(sample['name'], start=1): # REVERSE ORDER (so models are used in reverse order), note middle ones will still get flash-lite 
    # Reset Gemini model order if appropriate (after 1 minute)
    if time.time() - last_reset > 60:
        reset_model_order()
        
    print(f"Processing card: {card}. This is card {idx} / {len(sample)}.")
    new_row = None

    # First try using Gemini models if any remain
    if model_order:
        new_row = try_gemini_models(card)
    else:
        print("No Gemini models available. Falling back to local model.")
    
    # If Gemini models failed or none remain, try local model
    if new_row is None:
        try:
            print(f"Trying local model for card: {card}")
            response = categorize_card(card, local=True)
            new_row = parse_response(card, 'gemma3:latest', response, run)
            print(f"Local model success: {new_row['category'].iloc[0]}")
        except Exception as e:
            print(f"Local model failed for {card} with error: {e}. Skipping card.")
            continue
    
    # Add the successfully categorized card to results.
    if new_row is not None:
        results = pd.concat([results, new_row], ignore_index=True)

Resetting model_order to all available Gemini models after 1 minute.
Model order has been reset to: ['gemini-2.5-pro', 'gemini-2.5-flash', 'gemini-2.5-flash-lite']
Processing card: Performage Wind Drainer. This is card 1 / 1340.
Trying Gemini API with model: gemini-2.5-pro
Gemini API success: Extender
Processing card: D/D Scale Surveyor. This is card 2 / 1340.
Trying Gemini API with model: gemini-2.5-pro
Gemini API failed for D/D Scale Surveyor with model gemini-2.5-pro with error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}. Removing this model for this round.
Trying Gemini API with model: gemini-2.5-flash
Gemini API success: Extender
Processing card: Centur-Ion Auxila. This is card 3 / 1340.
Trying Gemini API with model: gemini-2.5-pro
Gemini API success: Extender
Processing card: Miasma Dragon Tistina. This is card 4 / 1340.
Trying Gemini API with model: gemini-2.5-pro
Gemini API success: Starter
P

In [11]:
# Save the results to a CSV file in ../Data/
results.to_csv("../Data/categorized_cards.csv", index=False)

# Manual Review

### Categorization Fix

In [70]:
# Redo any cards who aren't one of the expected categories
redo_cards = results[~results['category'].isin(['Starter', 'Handtrap', 'Extender', 
                                                'Board Breaker', 'Garnet', 'None'])].copy()
if not redo_cards.empty:
    print(f"Redoing {len(redo_cards)} cards that are not in the expected categories.")

# Check if the justification column has the category and pull it out using regex


Redoing 741 cards that are not in the expected categories.


In [ ]:
def find_category(row):
    ''' 
    Helper function to find the category in the row.
    It checks both the 'category' and 'justification' columns for a match. 
    '''
    # First, try to find the category in the 'category' column itself
    match = re.search(r'Category:\s*(\w+)', str(row['category']))
    if match:
        return match.group(1)
    
    # If not found, try to find it in the 'justification' column
    match = re.search(r'Category:\s*(\w+)', str(row['justification']))
    if match:
        return match.group(1)

    # Otherwise, return the original category
    return row['category']

def fix_board(row):
    ''' 
    Helper function to fix the category if it is 'Board'.
    This is a special case where 'Board' should be changed to 'Board Breaker'.
    '''
    # If the category is 'Board', change it to 'Board Breaker'
    if row['category'] == 'Board':
        return 'Board Breaker'
    return row['category']

def fix_category(row):
    ''' 
    Helper function to fix the category of a card.
    It checks the 'category' and 'justification' columns for the category,
    and applies the find_category and fix_board functions to ensure the category is correct.
    If the category is still not one of the expected categories, it sets it to 'None', which after evaluating the justification columns seems to be appropriate. A new prompt could be created to handle this better in the future as the models are not returning None as a category.
    '''
    row['category'] = find_category(row)
    row['category'] = fix_board(row)

    # If the category is still not one of the expected categories, set it to 'None'
    if row['category'] not in ['Starter', 'Handtrap', 'Extender',
                               'Board Breaker', 'Garnet', 'None']:
        return 'None'
    
    return row['category']


redo_cards['category'] = redo_cards.apply(fix_category, axis=1)

redo_cards

,card_id,card_name,category,justification,card_text,run,model
1,34541863,"""A"" Cell Breeding Device",None,"The card's effect is passive and slow, placing...",NaN,1,NaN
2,64163367,"""A"" Cell Incubator",None,This card is a highly specific support piece f...,NaN,1,NaN
5,98319530,"""Infernoble Arms - Almace""",Extender,"Justification: This card's effects, particular...",While this card is equipped to a monster: You ...,1,NaN
6,37478723,"""Infernoble Arms - Durendal""",Extender,Justification: The card's primary search effec...,While this card is equipped to a monster: You ...,1,NaN
7,64867422,"""Infernoble Arms - Hauteclere""",Extender,Justification: This card requires an already e...,While this card is equipped to a monster: You ...,1,NaN
...,...,...,...,...,...,...,...
2690,30655537,Cyber Falcon,None,The provided text does not contain any card ef...,A jet-powered hawk that travels at the speed o...,2,gemini-2.5-flash-lite
2692,28725004,Tainted Wisdom,None,This effect only changes the position of a car...,If this Attack Position card is changed to fac...,2,gemini-2.5-flash-lite
2693,14618326,Crimson Ninja,None,This card's effect is a simple Trap Card destr...,FLIP: Target 1 Trap Card on the field; destroy...,2,gemini-2.5-flash-lite
2694,473469,Driving Snow,None,This card requires a specific trigger conditio...,You can only activate this card when 1 or more...,2,gemini-2.5-flash-lite


Now that we are confident fix_category() works as intended, we can apply it to the category in the main dataframe

In [ ]:
results['category'] = results.apply(fix_category, axis=1)

(results['category'].value_counts(normalize=True) * 100).round(2).to_frame(name='Percentage').sort_index().reset_index().rename(columns={'index': 'Category'})

,category,Percentage
0,Board Breaker,10.24
1,Extender,34.64
2,Garnet,1.15
3,Handtrap,8.61
4,None,26.85
5,Starter,18.51


### Resolve Disagreements

In [93]:
results

,card_id,card_name,category,justification,card_text,run,model
0,80181649,"""A Case for K9""",Starter,"The primary effect, ""When this card is activated",NaN,1,NaN
1,34541863,"""A"" Cell Breeding Device",None,"The card's effect is passive and slow, placing...",NaN,1,NaN
2,64163367,"""A"" Cell Incubator",None,This card is a highly specific support piece f...,NaN,1,NaN
3,91231901,"""A"" Cell Recombination Device",Extender,"The card's primary effect sends an ""Alien"" mon...",NaN,1,NaN
4,73262676,"""A"" Cell Scatter Burst",Extender,"This card requires an ""Alien"" monster already ...",NaN,1,NaN
...,...,...,...,...,...,...,...
2691,32919136,Falling Down,Board Breaker,This card's effect allows the player to steal ...,Activate this card by targeting an opponent's ...,2,gemini-2.5-flash-lite
2692,28725004,Tainted Wisdom,None,This effect only changes the position of a car...,If this Attack Position card is changed to fac...,2,gemini-2.5-flash-lite
2693,14618326,Crimson Ninja,None,This card's effect is a simple Trap Card destr...,FLIP: Target 1 Trap Card on the field; destroy...,2,gemini-2.5-flash-lite
2694,473469,Driving Snow,None,This card requires a specific trigger conditio...,You can only activate this card when 1 or more...,2,gemini-2.5-flash-lite


In [ ]:
# Categories
categories = results.pivot_table(index=['card_id', 'card_name', 'card_text'],
                                columns='run', 
                                values='category',
                                aggfunc='first').reset_index()

categories.columns = [f"category_{col}" if isinstance(col, (int, float)) else col 
                         for col in categories.columns]

# Justifications with suffix
justifications = results.pivot_table(index=['card_id', 'card_name'],
                                    columns='run', 
                                    values='justification',
                                    aggfunc='first').reset_index()

# Rename justification columns to avoid conflicts
justifications.columns = [f"justification_{col}" if isinstance(col, (int, float)) else col 
                         for col in justifications.columns]

# Merge them
both_results = categories.merge(justifications, on=['card_id', 'card_name'])

both_results["agreed"] = both_results.apply(lambda row: row.iloc[3:5].nunique() == 1, axis=1)  

both_results.to_csv("../Data/categorized_cards_both_results.csv", index=False)

# RoBERTa Classifier